# QazRuFake: transformer baselines (XLM-R, mBERT)

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
Cell 2 will ask you to upload `data/qazrufake.jsonl` from this repository.

Each model is trained 3 times (seeds 1–3) on RU, KK and RU+KK, and every trained model is evaluated on both test sets. That covers in-language, zero-shot cross-lingual and joint settings in one pass. On a T4 this takes about 1–1.5 hours. Results are saved to Google Drive after every run, so nothing is lost if Colab disconnects.

In [ ]:
# 1. Check the GPU and install packages
import subprocess, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: switch Runtime -> T4 GPU, then rerun')
!pip install -q -U transformers datasets accelerate scikit-learn

In [ ]:
# 2. Upload corpus.jsonl and mount Drive for saving results
import os
from google.colab import files, drive
if not os.path.exists('qazrufake.jsonl'):
    up = files.upload()            # choose data/qazrufake.jsonl
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/qazrufake_results.jsonl'
import json
R = [json.loads(l) for l in open('qazrufake.jsonl', encoding='utf-8')]
print(len(R), 'items loaded; results ->', OUT)

In [ ]:
# 3. Training function
import json, shutil, numpy as np, torch
from datasets import Dataset
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)

def pick(langs, split):
    return [r for r in R if r['lang'] in langs and r['split'] == split]

def done(model, train, seed):
    if not os.path.exists(OUT): return False
    for l in open(OUT):
        d = json.loads(l)
        if d['model'] == model and d['train'] == train and d['seed'] == seed: return True
    return False

def run(model_name, train='ru', seed=1, epochs=4, lr=2e-5, bs=16, maxlen=256):
    if done(model_name, train, seed):
        print('skip (already done):', model_name, train, seed); return
    set_seed(seed)
    langs = train.split(',')
    tok = AutoTokenizer.from_pretrained(model_name)
    def ds(rs):
        d = Dataset.from_dict({'text': [r['text'] for r in rs], 'label': [r['label'] for r in rs]})
        return d.map(lambda b: tok(b['text'], truncation=True, max_length=maxlen), batched=True)
    dtr, ddev = ds(pick(langs, 'train')), ds(pick(langs, 'dev'))
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    def metrics(p):
        y = p.label_ids; pr = p.predictions.argmax(-1)
        return {'f1m': f1_score(y, pr, average='macro'), 'acc': accuracy_score(y, pr)}
    args = TrainingArguments(output_dir='out', learning_rate=lr, per_device_train_batch_size=bs,
        per_device_eval_batch_size=64, num_train_epochs=epochs, weight_decay=0.01, warmup_steps=int(0.1 * epochs * -(-len(dtr) // bs)),
        eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
        metric_for_best_model='f1m', save_total_limit=1, fp16=torch.cuda.is_available(),
        report_to='none', seed=seed)
    t = Trainer(model=model, args=args, train_dataset=dtr, eval_dataset=ddev,
                data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
    t.train()
    for test in ['ru', 'kk']:
        te = pick([test], 'test')
        p = t.predict(ds(te))
        y, logits = p.label_ids, p.predictions
        pr = logits.argmax(-1)
        prob = torch.softmax(torch.tensor(logits), -1)[:, 1].numpy()
        res = dict(model=model_name, train=train, test=test, seed=seed, n=len(te),
                   acc=float(accuracy_score(y, pr)), f1m=float(f1_score(y, pr, average='macro')),
                   f1fake=float(f1_score(y, pr)), auc=float(roc_auc_score(y, prob)))
        print(res)
        with open(OUT, 'a') as f: f.write(json.dumps(res) + '\n')
    del t, model; torch.cuda.empty_cache(); shutil.rmtree('out', ignore_errors=True)

In [ ]:
# 4. Run all experiments (about 1-1.5 h on T4). Safe to rerun: finished runs are skipped.
MODELS = ['xlm-roberta-base', 'bert-base-multilingual-cased']
for m in MODELS:
    for train in ['ru', 'kk', 'ru,kk']:
        for seed in [1, 2, 3]:
            print('=' * 20, m, train, seed)
            run(m, train, seed)

In [ ]:
# 5. Summary table: mean ± SD over seeds
import pandas as pd
df = pd.DataFrame([json.loads(l) for l in open(OUT)])
g = df.groupby(['model', 'train', 'test'])
tab = g[['acc', 'f1m', 'f1fake', 'auc']].agg(lambda s: f'{s.mean():.3f} ± {s.std():.3f}')
tab['seeds'] = g.size()
print(tab.to_string())
tab.to_csv('/content/drive/MyDrive/qazrufake_transformers_summary.csv')